# Geomar AI Oxygen - Hypoxia Prediction Training (Google Colab)

This notebook runs the weighted hypoxia prediction model training pipeline on Google Colab.

**Project**: Boknis Eck hypoxia prediction using weighted Temporal Fusion Transformer  
**Repository**: https://github.com/YOUR_USERNAME/Geomar_AI_Oxygen

## Notebook Overview

1. **Setup**: Install dependencies, clone repository, mount Google Drive
2. **Data Preparation**: Verify data pipeline
3. **Training Options**: Quick test / Full training / Hyperparameter tuning
4. **Results**: View metrics, download checkpoints

## Requirements

- GPU runtime REQUIRED (Runtime > Change runtime type > GPU)
- Google Drive for checkpoints (optional but recommended)
- **Standard RAM** is sufficient (DO NOT select High-RAM)

# 1. Setup Environment

In [1]:
# Check GPU availability and Colab pre-installed versions
import torch
import pandas as pd
import numpy as np

print("="*80)
print("SYSTEM INFO")
print("="*80)
print(f"\nPyTorch: {torch.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem_gb:.1f} GB")

    # Adjust batch size based on GPU
    if "A100" in gpu_name:
        RECOMMENDED_BATCH_SIZE = 128
        print(f"\n✓ A100 detected! Recommended batch size: {RECOMMENDED_BATCH_SIZE}")
    else:
        RECOMMENDED_BATCH_SIZE = 64
        print(f"\n✓ GPU detected! Recommended batch size: {RECOMMENDED_BATCH_SIZE}")
else:
    print("\n⚠️  NO GPU DETECTED - Training will be VERY slow!")
    print("   Go to: Runtime > Change runtime type > Hardware accelerator > GPU")
    RECOMMENDED_BATCH_SIZE = 32

SYSTEM INFO

PyTorch: 2.11.0+cu128
Pandas: 2.2.3
NumPy: 2.0.2

CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB

✓ GPU detected! Recommended batch size: 64


In [3]:
# Clone repository
import os

REPO_URL = "https://github.com/MaeTobiGeri/Geomar_AI_Oxygen.git"
REPO_NAME = "Geomar_AI_Oxygen"

if os.path.exists(REPO_NAME):
    print(f"Repository exists. Pulling latest changes...")
    !cd {REPO_NAME} && git pull
else:
    print(f"Cloning from {REPO_URL}...")
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
print(f"\n✓ Working directory: {os.getcwd()}")

Cloning from https://github.com/MaeTobiGeri/Geomar_AI_Oxygen.git...
Cloning into 'Geomar_AI_Oxygen'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 65 (delta 17), reused 61 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 3.54 MiB | 36.26 MiB/s, done.
Resolving deltas: 100% (17/17), done.

✓ Working directory: /content/Geomar_AI_Oxygen


In [4]:
# Install dependencies (Colab-compatible versions)
print("Installing dependencies...\n")
print("Note: Some version warnings are expected and safe to ignore.\n")

# Use Colab-compatible requirements
!pip install -q pytorch-forecasting==1.7.0
!pip install -q 'lightning>=2.0.0,<2.7.0'
!pip install -q 'wetterdienst>=0.90.0'
!pip install -q 'plotly>=5.0.0'
!pip install -q 'optuna>=3.0.0'

print("\n" + "="*80)
print("VERIFYING INSTALLATION")
print("="*80 + "\n")

try:
    import pytorch_forecasting
    import lightning.pytorch as pl
    import optuna
    from wetterdienst import Wetterdienst

    print("✓ All key packages imported successfully!\n")
    print(f"Package versions:")
    print(f"  pytorch-forecasting: {pytorch_forecasting.__version__}")
    print(f"  lightning: {pl.__version__}")
    print(f"  optuna: {optuna.__version__}")
    print(f"  torch: {torch.__version__}")
    print(f"  pandas: {pd.__version__}")
    print(f"  numpy: {np.__version__}")

except ImportError as e:
    print(f"\n❌ Import error: {e}")
    print("\nTry restarting runtime: Runtime > Restart runtime")

Installing dependencies...

Note: Some version warnings are expected and safe to ignore.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 441.2/441.2 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.1/847.1 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
from google.colab import drive

MOUNT_DRIVE = True

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/Geomar_Checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"\n✓ Checkpoints will be saved to Google Drive: {CHECKPOINT_DIR}")
    print("  (These will persist even after session ends)")
else:
    CHECKPOINT_DIR = 'models/hypoxia_tft'
    print(f"\n⚠️  Checkpoints will be saved locally: {CHECKPOINT_DIR}")
    print("  (These will be LOST when session ends! Download before closing.)")

Mounted at /content/drive

✓ Checkpoints will be saved to Google Drive: /content/drive/MyDrive/Geomar_Checkpoints
  (These will persist even after session ends)


# 2. Verify Data and Pipeline

In [6]:
# Verify data files and pipeline
print("Checking data files...\n")
!ls -lh Documentation/data/

print("\nTesting data ingestion...")
from src import data_ingestion

df_ocean = data_ingestion._load_ocean_data()
print(f"\n✓ Ocean data loaded: {len(df_ocean)} rows")
print(f"  Date range: {df_ocean['Date'].min()} to {df_ocean['Date'].max()}")
print(f"  Depths: {sorted(df_ocean['Depth_m'].unique())}")

Checking data files...

total 524K
-rw-r--r-- 1 root root 433K Aug 21 15:03 BoknisEck_1957-2014.csv
-rw-r--r-- 1 root root  59K Aug 21 15:03 BoknisEck_2015-2023.csv
-rw-r--r-- 1 root root  28K Aug 21 15:03 BoknisEck_chl_2015-2021.tab

Testing data ingestion...

✓ Ocean data loaded: 5964 rows
  Date range: 1957-04-30 00:00:00 to 2023-12-06 10:08:48
  Depths: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(35)]


In [7]:
# Run unit tests
print("Running unit tests...\n")
!pip install -q pytest
!python -m pytest tests/ -v --tb=short

print("\n✓ All tests passed! Pipeline is ready for training.")

Running unit tests...

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/Geomar_AI_Oxygen
plugins: anyio-4.14.2, typeguard-4.6.0, langsmith-0.11.0
collected 24 items                                                             

tests/test_data_ingestion.py::test_old_ocean_data_converts_oxygen_from_umol_per_kg PASSED [  4%]
tests/test_data_ingestion.py::test_new_ocean_data_keeps_oxygen_already_in_umol_per_liter PASSED [  8%]
tests/test_data_ingestion.py::test_chlorophyll_supplement_fills_gaps_but_does_not_override PASSED [ 12%]
tests/test_dataset.py::test_split_train_validation_is_chronological PASSED [ 16%]
tests/test_dataset.py::test_split_train_validation_preserves_order PASSED [ 20%]
tests/test_dataset.py::test_get_held_out_events_identifies_episodes PASSED [ 25%]
tests/test_dataset.py::test_get_held_out_events_filters_by_validatio

# 3. Training Options\n\n**Choose ONE of the following:**\n- Option A: Quick Test (5 epochs, 2-5 min)\n- Option B: Full Training (100 epochs with early stopping, 10-60 min depending on GPU)\n- Option C: Hyperparameter Tuning (LONG: 1-20 hours depending on GPU and trials)\n- Option D: Weighted Loss Verification (5-10 min)

## Option A: Quick Test Training (5 epochs)

In [8]:
# Quick test - verify everything works
print("Starting quick test (5 epochs)...\n")

!python train.py \
    --max-epochs 5 \
    --batch-size 32 \
    --checkpoint-path "{CHECKPOINT_DIR}" \
    --patience 2

print("\n✓ Quick test complete!")

Starting quick test (5 epochs)...

Seed set to 42
WEIGHTED HYPOXIA PREDICTION MODEL TRAINING

Hyperparameters:
  hidden_size: 16
  attention_head_size: 1
  dropout: 0.1
  hidden_continuous_size: 8
  learning_rate: 0.03
  lstm_layers: 1
  gradient_clip_val: 0.1

Dataset configuration:
  Encoder length: 8 weeks
  Decoder length: 4 weeks
  Batch size: 32
  Train/val split: 80%/20%

Training configuration:
  Max epochs: 5
  Early stopping patience: 2
  Checkpoint path: /content/drive/MyDrive/Geomar_Checkpoints

--------------------------------------------------------------------------------
Phase 2: Data Ingestion
--------------------------------------------------------------------------------
Ocean data loaded: 5964 rows
Traceback (most recent call last):
  File "/content/Geomar_AI_Oxygen/train.py", line 344, in <module>
    main()
  File "/content/Geomar_AI_Oxygen/train.py", line 190, in main
    df_weather = data_ingestion._load_weather_data()
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^

## Option B: Full Training

In [ ]:
# Full training with default hyperparameters
print(f"Starting full training (batch_size={RECOMMENDED_BATCH_SIZE})...\n")

!python train.py \
    --max-epochs 100 \
    --batch-size {RECOMMENDED_BATCH_SIZE} \
    --checkpoint-path "{CHECKPOINT_DIR}" \
    --patience 3

print("\n✓ Full training complete!")

## Option C: Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning with Optuna
N_TRIALS = 20  # Increase to 50 for production

print(f"Starting hyperparameter tuning ({N_TRIALS} trials)...")
print("⏱️  This will take 1-5 hours depending on GPU.\n")

!python tune_hyperparameters.py \
    --n-trials {N_TRIALS} \
    --output tuned_hyperparameters.json

# Show best hyperparameters
import json
with open('tuned_hyperparameters.json') as f:
    tuned = json.load(f)

print("\n" + "="*80)
print("BEST HYPERPARAMETERS")
print("="*80)
for key, value in tuned['hyperparameters'].items():
    print(f"  {key}: {value}")
print(f"\nBest val_loss: {tuned['val_loss']:.4f}")

In [ ]:
# Train with tuned hyperparameters
print("Training with tuned hyperparameters...\n")

!python train.py \
    --load-hyperparameters tuned_hyperparameters.json \
    --checkpoint-path "{CHECKPOINT_DIR}" \
    --max-epochs 100 \
    --patience 3

print("\n✓ Training with tuned hyperparameters complete!")

## Option D: Weighted Loss Verification

In [ ]:
# Verify weighted loss mechanism
!python verify_weighted_loss.py

# 4. View Results

In [ ]:
# View training metadata
import json
from pathlib import Path

metadata_path = Path(CHECKPOINT_DIR) / "training_metadata.json"

if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)

    print("="*80)
    print("TRAINING METADATA")
    print("="*80)
    print(f"\nTraining Date: {metadata['training_date']}")

    print("\nHyperparameters:")
    for key, value in metadata['hyperparameters'].items():
        print(f"  {key}: {value}")

    print(f"\nDataset: {metadata['dataset_info']['total_samples']} samples")
    print(f"  Train: {metadata['dataset_info']['train_samples']}")
    print(f"  Val: {metadata['dataset_info']['val_samples']}")

    print(f"\nFeatures ({len(metadata['features'])}):")
    for feat in metadata['features']:
        print(f"  - {feat}")
else:
    print("⚠️  No training metadata found. Run training first.")

In [ ]:
# Download checkpoint (if not using Google Drive)
if not MOUNT_DRIVE:
    from google.colab import files

    print("Downloading checkpoints...\n")

    best_ckpt = Path(CHECKPOINT_DIR) / "best_model.ckpt"
    if best_ckpt.exists():
        files.download(str(best_ckpt))
        print("✓ Downloaded: best_model.ckpt")

    if metadata_path.exists():
        files.download(str(metadata_path))
        print("✓ Downloaded: training_metadata.json")
else:
    print(f"✓ Checkpoints saved to Google Drive: {CHECKPOINT_DIR}")
    print("  Access them anytime from your Drive!")

# Training Time Estimates\n\n**T4 GPU (Colab Free):**\n- Quick test: 5-10 min\n- Full training: 40-80 min\n- Tuning (20 trials): 4-8 hours\n\n**A100 GPU (Colab Pro+):**\n- Quick test: 2-3 min\n- Full training: 10-20 min\n- Tuning (20 trials): 1-2 hours\n\n**Tips:**\n- Start with Option A (quick test) first\n- Use Google Drive to preserve checkpoints\n- Monitor GPU: `!nvidia-smi`